# 06 - Pairwise judging with a Kaggle model

Create this notebook as a Kaggle utility benchmark notebook so `kaggle_benchmarks` can reach the model proxy.
Inputs: the outputs of notebooks 04 and 05. No GPU needed. Judging runs only on saved pairwise files.

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO = "small-model-alignment"
GITHUB_USER = "YOUR_GITHUB_USERNAME"
os.chdir("/kaggle/working")
if not os.path.exists(REPO):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    result = subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"], capture_output=True, text=True)
    print("cloned" if result.returncode == 0 else result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GITHUB_USER}/{REPO}.git"])
os.chdir(f"/kaggle/working/{REPO}")
os.environ["PYTHONPATH"] = "."
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!git log --oneline -3

In [ ]:
import glob
import shutil

import kaggle_benchmarks as kbench

os.makedirs("outputs/eval", exist_ok=True)
for path in glob.glob("/kaggle/input/**/outputs/eval/*_pairs.jsonl", recursive=True):
    shutil.copy(path, "outputs/eval/")
print(sorted(os.listdir("outputs/eval")))
print(list(kbench.llms))

In [ ]:
JUDGE_MODEL = "google/gemini-3.6-flash"
with kbench.chats.new("smoke-test"):
    print(kbench.llms[JUDGE_MODEL].prompt("Reply with the single word: ready"))

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/hh_dpo_vs_sft_pairs.jsonl --output_path outputs/eval/hh_dpo_vs_sft_judged.jsonl --judge_model {JUDGE_MODEL} --rubric safety --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/alpaca_dpo_uf_vs_sft_pairs.jsonl --output_path outputs/eval/alpaca_dpo_uf_vs_sft_judged.jsonl --judge_model {JUDGE_MODEL} --rubric general --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/persona_lora_vs_instruct_pairs.jsonl --output_path outputs/eval/persona_lora_vs_instruct_style_judged.jsonl --judge_model {JUDGE_MODEL} --rubric persona --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/persona_lora_vs_instruct_pairs.jsonl --output_path outputs/eval/persona_lora_vs_instruct_helpful_safe_judged.jsonl --judge_model {JUDGE_MODEL} --rubric safety --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

Secondary analysis: does safety-oriented DPO change general helpfulness on AlpacaEval?

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/alpaca_dpo_hh_vs_sft_pairs.jsonl --output_path outputs/eval/alpaca_dpo_hh_vs_sft_judged.jsonl --judge_model {JUDGE_MODEL} --rubric general --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
!cd /kaggle/working && zip -r -q judge_outputs.zip small-model-alignment/outputs/eval
!ls -lh outputs/eval